In [ ]:
import pandas as pd
from google.colab import drive

# Connect Google Drive so the dataset survives disconnects
drive.mount("/content/drive")

# Load the dataset from the churn_project folder in Drive
df = pd.read_csv("/content/drive/MyDrive/churn_project/WA_Fn-UseC_-Telco-Customer-Churn.csv")

print(df.shape)
df.head()

Mounted at /content/drive
(7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [ ]:
df.info()

print("\nMissing values:")
print(df.isnull().sum())

print("\nChurn counts:")
print(df["Churn"].value_counts())
print(df["Churn"].value_counts(normalize=True).round(3))


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [ ]:
# Fix TotalCharges: it's stored as text, so convert it to numbers
print("Before:", df["TotalCharges"].dtype)

# Blank values can't be converted, so errors="coerce" turns them into NaN
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
print("After:", df["TotalCharges"].dtype)

# Count how many values are missing now
print("Missing in TotalCharges:", df["TotalCharges"].isnull().sum())

# Check who these customers are
print(df[df["TotalCharges"].isnull()][["tenure", "MonthlyCharges", "TotalCharges"]])

Before: object
After: float64
Missing in TotalCharges: 11
      tenure  MonthlyCharges  TotalCharges
488        0           52.55           NaN
753        0           20.25           NaN
936        0           80.85           NaN
1082       0           25.75           NaN
1340       0           56.05           NaN
3331       0           19.85           NaN
3826       0           25.35           NaN
4380       0           20.00           NaN
5218       0           19.70           NaN
6670       0           73.35           NaN
6754       0           61.90           NaN


In [ ]:
# These 11 customers just joined (tenure = 0), so their total charges are 0
df["TotalCharges"] = df["TotalCharges"].fillna(0)

# Confirm no missing values are left
print("Missing after fix:", df["TotalCharges"].isnull().sum())


Missing after fix: 0


In [ ]:
# Turn Churn into 1/0 so we can average it to get a churn rate
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

# Average of 1/0 = share of customers who churned, per contract type
print(df.groupby("Contract")["Churn"].mean().round(3))

Contract
Month-to-month    0.427
One year          0.113
Two year          0.028
Name: Churn, dtype: float64


### Finding 1: Contract type
Month-to-month customers churn at 42.7%, compared to 2.8% for two-year contracts.
Customers with no long-term commitment are much more likely to leave.

In [ ]:
# Put each customer into a tenure group (like age groups)
df["tenure_group"] = pd.cut(
    df["tenure"],
    bins=[-1, 12, 24, 48, 72],
    labels=["0-12 months", "13-24 months", "25-48 months", "49-72 months"],
)

# Churn rate for each group
print(df.groupby("tenure_group", observed=True)["Churn"].mean().round(3))

tenure_group
0-12 months     0.474
13-24 months    0.287
25-48 months    0.204
49-72 months    0.095
Name: Churn, dtype: float64


### Finding 2: Tenure
Customers with short tenure churn the most, and churn drops steadily as tenure grows.
The first year is the riskiest period for losing a customer.

### Finding 3: Monthly charges
Customers who churned paid slightly higher monthly charges on average than those who stayed.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score

# X = the clues about each customer, y = what we want to predict (churn)
# customerID is just a label, and tenure_group is made from tenure, so both are dropped
X = df.drop(columns=["customerID", "Churn", "tenure_group"])
y = df["Churn"]

# Numbers get scaled, text columns get converted to 0/1 columns
num_cols = ["tenure", "MonthlyCharges", "TotalCharges"]
cat_cols = [c for c in X.columns if c not in num_cols]

# Keep 20% of customers aside to test the model on data it hasn't seen
# stratify=y keeps the same churn percentage in both parts
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Preparing the data: scale the numbers, one-hot encode the text columns
prep = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
])

# Pipeline = prep + model in one step
# class_weight="balanced" because only about 26% of customers churn
model = Pipeline([
    ("prep", prep),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced")),
])

# Train on 80% of the data
model.fit(X_train, y_train)

# Test on the 20% the model has never seen
preds = model.predict(X_test)
probs = model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, preds))
print("ROC-AUC:", round(roc_auc_score(y_test, probs), 3))

              precision    recall  f1-score   support

           0       0.90      0.72      0.80      1035
           1       0.50      0.78      0.61       374

    accuracy                           0.74      1409
   macro avg       0.70      0.75      0.71      1409
weighted avg       0.80      0.74      0.75      1409

ROC-AUC: 0.842


In [ ]:
# Get the column names after one-hot encoding
feature_names = model.named_steps["prep"].get_feature_names_out()

# Get the weight the model learned for each column
weights = model.named_steps["clf"].coef_[0]

# Put them together and sort from most "stay" to most "churn"
importance = pd.Series(weights, index=feature_names).sort_values()

print("Pushes toward STAYING:")
print(importance.head(8).round(2))

print("\nPushes toward CHURNING:")
print(importance.tail(8).round(2))

Pushes toward STAYING:
num__tenure                                 -1.14
cat__Contract_Two year                      -0.77
num__MonthlyCharges                         -0.68
cat__InternetService_DSL                    -0.62
cat__OnlineSecurity_No internet service     -0.27
cat__OnlineBackup_No internet service       -0.27
cat__InternetService_No                     -0.27
cat__DeviceProtection_No internet service   -0.27
dtype: float64

Pushes toward CHURNING:
cat__TechSupport_No                    0.17
cat__OnlineSecurity_No                 0.21
cat__PaymentMethod_Electronic check    0.24
cat__StreamingTV_Yes                   0.27
cat__StreamingMovies_Yes               0.28
num__TotalCharges                      0.47
cat__Contract_Month-to-month           0.66
cat__InternetService_Fiber optic       0.72
dtype: float64


### Finding 4: What the model learned
The strongest signals for staying are long tenure and a two-year contract.
The strongest signals for churning are fiber optic internet and month-to-month contracts.
Customers without tech support or online security also churn slightly more.
Note: tenure, MonthlyCharges and TotalCharges overlap, so their individual weights should be read with care.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import recall_score, precision_score

# Random Forest: 300 trees, each leaf needs at least 5 customers
# (this stops the trees from memorizing the training data)
rf_model = Pipeline([
    ("prep", prep),
    ("clf", RandomForestClassifier(
        n_estimators=300,
        min_samples_leaf=5,
        class_weight="balanced",
        random_state=42,
    )),
])

# Train on the same 80% of customers as the first model
rf_model.fit(X_train, y_train)

# Score both models on the same 20% test set
rows = []
for name, m in [("Logistic Regression", model), ("Random Forest", rf_model)]:
    preds = m.predict(X_test)
    probs = m.predict_proba(X_test)[:, 1]
    rows.append({
        "Model": name,
        "ROC-AUC": round(roc_auc_score(y_test, probs), 3),
        "Recall (churners)": round(recall_score(y_test, preds), 3),
        "Precision (churners)": round(precision_score(y_test, preds), 3),
    })

print(pd.DataFrame(rows))

                 Model  ROC-AUC  Recall (churners)  Precision (churners)
0  Logistic Regression    0.842              0.783                 0.504
1        Random Forest    0.842              0.749                 0.552


In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

# Cutoffs to try: customers above the cutoff are flagged as "will churn"
thresholds = [0.3, 0.4, 0.5, 0.6, 0.7]

rows = []
for name, m in [("Logistic Regression", model), ("Random Forest", rf_model)]:
    # Churn probability for each customer in the test set
    probs = m.predict_proba(X_test)[:, 1]

    for t in thresholds:
        # Flag as churn (1) if the probability is at or above the cutoff
        preds = (probs >= t).astype(int)
        rows.append({
            "Model": name,
            "Cutoff": t,
            "Recall": round(recall_score(y_test, preds), 3),
            "Precision": round(precision_score(y_test, preds), 3),
            "F1": round(f1_score(y_test, preds), 3),
            "Flagged": int(preds.sum()),
        })

print(pd.DataFrame(rows))

                 Model  Cutoff  Recall  Precision     F1  Flagged
0  Logistic Regression     0.3   0.928      0.429  0.587      808
1  Logistic Regression     0.4   0.866      0.467  0.607      694
2  Logistic Regression     0.5   0.783      0.504  0.614      581
3  Logistic Regression     0.6   0.706      0.540  0.612      489
4  Logistic Regression     0.7   0.602      0.603  0.602      373
5        Random Forest     0.3   0.880      0.456  0.601      721
6        Random Forest     0.4   0.816      0.510  0.628      598
7        Random Forest     0.5   0.749      0.552  0.636      507
8        Random Forest     0.6   0.607      0.596  0.601      381
9        Random Forest     0.7   0.471      0.672  0.553      262


In [ ]:
import joblib

# Save the Random Forest pipeline (prep steps + model) to Drive
joblib.dump(rf_model, "/content/drive/MyDrive/churn_project/churn_model.pkl")

# The cutoff we chose is saved separately, since the model file doesn't store it
with open("/content/drive/MyDrive/churn_project/cutoff.txt", "w") as f:
    f.write("0.4")

print("Saved model and cutoff")

Saved model and cutoff


In [ ]:
import sklearn, sys

# We'll install this exact scikit-learn version on the laptop
print("scikit-learn:", sklearn.__version__)
print("Python:", sys.version.split()[0])

scikit-learn: 1.6.1
Python: 3.13.18


In [ ]:
import pandas as pd
from sklearn.inspection import permutation_importance

# Shuffle each original column on the test set, 10 times,
# and measure how much the ROC-AUC score drops
result = permutation_importance(
    rf_model, X_test, y_test,
    scoring="roc_auc", n_repeats=10, random_state=42,
)

# Bigger drop = the model depends on that column more
drops = pd.DataFrame({
    "Clue": X_test.columns,
    "Importance": result.importances_mean,
}).sort_values("Importance", ascending=False)

print(drops.head(10).round(3))

# Save the top 10 next to the model so the dashboard can read them
drops.head(10).to_csv("/content/drive/MyDrive/churn_project/importance.csv", index=False)
print("Saved importance.csv")

                Clue  Importance
14          Contract       0.052
4             tenure       0.026
7    InternetService       0.014
18      TotalCharges       0.014
8     OnlineSecurity       0.005
11       TechSupport       0.004
16     PaymentMethod       0.002
15  PaperlessBilling       0.002
1      SeniorCitizen       0.002
9       OnlineBackup       0.001
Saved importance.csv
